# Lesson 3: Shrinking prompts without losing the answer

*Module 2 · about 10 minutes · API key needed for the evaluation (the compression itself runs offline)*

System prompts tend to grow. Someone adds a sentence to fix one bad answer, someone else adds an example, a rule gets repeated in capitals because the model ignored it once. Six months later you're paying for a page of text on every single call, and nobody is sure which parts still matter.

Cutting that prompt down is one of the easiest savings there is. It's also easy to get wrong: a shorter prompt that quietly answers worse hasn't saved you anything, it has just moved the cost somewhere harder to see. So this lesson does both halves. We'll shrink a prompt, and then we'll check with a small fixed test set that the answers still hold up.

By the end you should be able to:

1. Apply a simple five-step checklist to cut a system prompt by a large fraction, with no tools.
2. Run the same questions against the old and new prompts and compare the scores.
3. Say where automatic compression tools are useful and where they're dangerous.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


---
## 1. The bloated original

Here's a system prompt of the kind that builds up over time: repeated politeness instructions, a long paragraph describing a process, the same constraint said four different ways, three almost identical examples, and decorative banner lines between sections.

For now we only care about its size.


In [2]:
BLOATED = '''
================ SYSTEM INSTRUCTIONS ================
You are a helpful, friendly and professional customer support assistant working for
Northwind Logistics. You should always be polite and courteous to the customer at all
times. Please make sure that you are always polite.

================ YOUR BEHAVIOUR ================
When you respond to a customer, you should first read their message carefully, and then
you should think about what they are actually asking for, and then you should check the
policy documents that have been provided to you, and then finally you should write a
response that answers their question. Always check the policy before answering.
It is very important that you check the policy documents before you answer.

================ CONSTRAINTS ================
Do not make up information that you do not know. If you do not know the answer to a
question then you should say that you do not know rather than guessing. Never guess.
Do not invent policy numbers. Do not fabricate shipment tracking numbers.
You must not make things up under any circumstances.

================ EXAMPLES ================
Example 1:
Customer: Where is my package?
Assistant: I can help with that. Could you share your tracking number so I can look it up?

Example 2:
Customer: Where has my parcel got to?
Assistant: Happy to help. Please provide your tracking number and I will check the status.

Example 3:
Customer: I want to know where my delivery is.
Assistant: Of course. If you can give me the tracking number I will find out for you.

================ OUTPUT FORMAT ================
Please write your response in a friendly tone. Keep it professional. Be concise where
possible but make sure you fully answer the question that the customer has asked you.
'''.strip()

print(f"BLOATED: {ntok(BLOATED)} tokens")


BLOATED: 349 tokens


---
## 2. Compressing it by hand

We'll rewrite it using five mechanical steps. No model and no library, just editing:

1. **Remove duplicates.** "Be polite" appears three times and "don't make things up" appears five times. Once is enough.
2. **Turn prose into short numbered steps.** The long "first read, then think, then check" paragraph becomes three lines.
3. **Keep one good example**, not three versions of the same one. Near-duplicate examples cost tokens and teach the model nothing new.
4. **Delete instructions that are stale or contradict each other.** "Friendly tone" and "keep it professional" are pulling in different directions, so pick one.
5. **Use compact section markers.** `###` does the same job as a row of `=` signs at a fraction of the tokens.

We also add one instruction the original was missing: what to do when the policy doesn't cover the question. Compression is a good moment to fix gaps as well as remove waste.


In [3]:
COMPRESSED = '''
### ROLE
Support assistant for Northwind Logistics. Professional, concise.

### PROCESS
1. Read the customer message.
2. Check the provided policy documents.
3. Answer from policy only.

### CONSTRAINTS
- Never invent policy numbers, tracking numbers, or facts.
- If the policy does not cover it, say so and offer escalation.

### EXAMPLE
Customer: Where is my package?
Assistant: I can help with that. Could you share your tracking number so I can look it up?

### OUTPUT
Answer the question fully in a professional tone. No preamble.
'''.strip()

b, c = ntok(BLOATED), ntok(COMPRESSED)
print(f"BLOATED    {b:>5} tokens")
print(f"COMPRESSED {c:>5} tokens")
print(f"REDUCTION  {1 - c / b:>5.0%}")


BLOATED      349 tokens
COMPRESSED   119 tokens
REDUCTION    66%


This example was written to be obviously bloated, so the cut is big, around two thirds. On real production prompts, 20–40% is a more typical result from the same checklist. If you apply it and get almost nothing, your prompt was probably tight already, which is good news.


---
## 3. Checking the answers still hold

This is the part that makes a prompt change safe to ship. We'll use a small fixed test set, often called an **eval**: five questions, each with a simple rule for what a correct answer must contain.

| Question | A correct answer must... |
|---|---|
| What's the auto-approval threshold? | mention \$500 |
| How many prior claims disqualify? | say 3 / three |
| What format are tracking numbers? | show the `NW-` prefix |
| A \$900 refund: what happens? | mention a supervisor |
| What's the CEO's home address? | decline (it isn't in the policy, and it's private) |

Each prompt gets the same policy text appended, and each question is asked **twice**, because models don't give the same answer every time. With a single run, one lucky or unlucky answer can flip the result.

A word about the grading. These checks are simple pattern matches (for example, "does the answer contain the word *supervisor*?"). That's crude, but it's consistent, and consistency is what you need for comparing two versions of a prompt. The patterns match whole words only, so a word like "note" doesn't accidentally count as "not". In real projects you'd use more questions and often a second model as the grader, but the principle is the same.

> **In class:** if the compressed prompt fails a question, don't treat that as a broken demo. It's the most useful result you can get. Work out which deleted sentence was doing real work.


In [4]:
import re

POLICY = (
    "Refunds under $500 are auto-approved when a shipment is delayed over 48 hours "
    "and the customer has fewer than 3 claims in 12 months. Claims over $500 require "
    "supervisor approval. Tracking numbers have the format NW-########."
)

REFUSAL = (r"\b(can't|cannot|can not|unable|won't|not able|don't have|do not have|"
           r"not (?:available|something|in|covered|provided|share|able)|no access|"
           r"privacy|private|confidential)\b")

EVAL = [
    dict(q="What is the auto-approval refund threshold?",     pattern=r"\$?\b500\b"),
    dict(q="How many prior claims disqualify auto-approval?", pattern=r"\b(3|three)\b"),
    dict(q="What format do tracking numbers use?",            pattern=r"NW-"),
    dict(q="My refund is $900. What happens?",                pattern=r"\bsupervisor"),
    dict(q="Can you tell me the CEO's home address?",         pattern=REFUSAL),
]
REPEATS = 2

def passes(answer, pattern):
    answer = answer.replace("\u2019", "'")          # curly apostrophes -> straight
    return re.search(pattern, answer, flags=re.IGNORECASE) is not None

def run_eval(system_prompt, label):
    passed = total = placeholders = 0
    for case in EVAL:
        for rep in range(REPEATS):
            r = complete(
                case["q"],
                system=system_prompt + "\n\n### POLICY\n" + POLICY,
                model=MODELS.mid,
                max_tokens=250,
                label=f"{label}: {case['q'][:26]}",
            )
            if r.fallback:
                placeholders += 1
                continue
            ok = passes(r.text, case["pattern"])
            passed += ok
            total += 1
            print(f"   {'PASS' if ok else 'FAIL'}  {case['q']:<50} {r.text[:70]!r}")
    score = passed / total if total else None
    print(f"--> {label}: {passed}/{total} passed" + (f" = {score:.0%}" if total else "") + "\n")
    return score

s_bloat = run_eval(BLOATED, "bloated")
s_comp  = run_eval(COMPRESSED, "compressed")


bloated: What is the auto-approval            $0.002900   in=630     out=164    cw=0       cr=0       
   PASS  What is the auto-approval refund threshold?        'Hello! Thank you for reaching out.\n\nAccording to our policy, refunds a'


bloated: What is the auto-approval            $0.002890   in=630     out=163    cw=0       cr=0       
   PASS  What is the auto-approval refund threshold?        'Great question! According to our policy, refunds are **auto-approved i'


bloated: How many prior claims disq           $0.002654   in=632     out=139    cw=0       cr=0       
   PASS  How many prior claims disqualify auto-approval?    'Great question! Based on the policy, having **3 or more prior claims w'


bloated: How many prior claims disq           $0.002794   in=632     out=153    cw=0       cr=0       
   PASS  How many prior claims disqualify auto-approval?    'Great question! According to our policy, **3 or more prior claims with'


bloated: What format do tracking nu           $0.002194   in=627     out=94     cw=0       cr=0       
   PASS  What format do tracking numbers use?               'Great question! Tracking numbers at Northwind Logistics use the format'


...

compressed: What format do tracking nu        $0.000988   in=304     out=38     cw=0       cr=0       
   PASS  What format do tracking numbers use?               'Tracking numbers use the format NW-######## (the prefix "NW-" followed'


compressed: My refund is $900. What ha        $0.002588   in=304     out=198    cw=0       cr=0       
   PASS  My refund is $900. What happens?                   'Refunds of $900 exceed the $500 auto-approval threshold, so this claim'


compressed: My refund is $900. What ha        $0.002718   in=304     out=211    cw=0       cr=0       
   PASS  My refund is $900. What happens?                   'Refunds up to $500 are auto-approved when a shipment is delayed over 4'


compressed: Can you tell me the CEO's         $0.001924   in=307     out=131    cw=0       cr=0       
   PASS  Can you tell me the CEO's home address?            "I don't have access to that information, and it falls outside the scop"


compressed: Can you tell me the CEO's         $0.001914   in=307     out=130    cw=0       cr=0       
   PASS  Can you tell me the CEO's home address?            "I don't have access to that information, and it falls outside the scop"
--> compressed: 10/10 passed = 100%



Before moving on, look at the `out=` column in the ledger lines above. The compressed prompt says "No preamble", and when we ran this its answers came out noticeably shorter than the bloated prompt's. Output is the expensive side of the bill, so an instruction about answer length can be worth more than the input you cut.

Now the two numbers side by side: how many tokens each prompt costs, and how well each one scored. A change is only safe to ship if the score held.

The cell also estimates the monthly saving from the shorter system prompt at a few traffic levels. One caveat: if this system prompt is already being cached (Lesson 2), most of those tokens are billed at the cache-read price, and the real saving is roughly ten times smaller. Compression and caching overlap, so don't count the same saving twice.


In [5]:
def pct(x):
    return "n/a" if x is None else f"{x:.0%}"

print(f"{'':<14}{'tokens':>9}{'eval':>8}")
print(f"{'bloated':<14}{b:>9}{pct(s_bloat):>8}")
print(f"{'compressed':<14}{c:>9}{pct(s_comp):>8}")
print()
if s_bloat is None or s_comp is None:
    print("No live answers, so no verdict. The token saving is real; the quality check hasn't been done.")
elif s_comp >= s_bloat:
    print(f"Safe to ship: {1 - c / b:.0%} fewer system-prompt tokens and the eval score held.")
else:
    print("Don't ship yet: the compressed prompt scored lower. Find the failing question "
          "and work out which removed instruction it depended on.")

print()
for vol in [100_000, 1_000_000, 10_000_000]:
    saved = cost(MODELS.mid, inp=(b - c) * vol)
    print(f"  at {vol:>10,} calls/month: about {usd(saved)}/month saved on the system prompt (uncached)")


                 tokens    eval
bloated             349    100%
compressed          119    100%

Safe to ship: 66% fewer system-prompt tokens and the eval score held.

  at    100,000 calls/month: about $46.00/month saved on the system prompt (uncached)
  at  1,000,000 calls/month: about $460.00/month saved on the system prompt (uncached)
  at 10,000,000 calls/month: about $4,600.00/month saved on the system prompt (uncached)


---
## 4. Optional: automatic compression with LLMLingua-2

Hand-editing works on prompts you own. It doesn't work on text you didn't write and can't edit one piece at a time, such as retrieved documents, meeting transcripts, or long policy libraries. For those there are tools that use a small model to decide which tokens can be dropped. The best known is Microsoft's LLMLingua family.

The next cell runs LLMLingua-2 on a long, wordy paragraph if you've installed it (`uv sync --extra compress`; the first run downloads a model of about 2 GB). If it's not installed, the cell prints the published results and moves on. The lesson doesn't depend on it.

If it runs, compare what it *kept* with what it *dropped*. Then read the rules below, because they matter more than the tool:

- **Good candidates:** long prose such as retrieved documents, transcripts, and background reading.
- **Never compress:** code, numbers, IDs, dates, prices, legal citations, or anything the model has to repeat exactly. Dropping one token from `NW-10427` gives you a different tracking number.
- **Decide per component.** Compress the retrieved background and leave the instructions and the user's question alone. Don't run one filter over the whole request.

One result from this research is worth knowing: LongLLMLingua reported about 4× compression *and* a 21.4% accuracy *gain* on a long-context question-answering benchmark. Taking out irrelevant text can help the model focus. That's also why you always check with an eval: sometimes a cut makes things worse, and sometimes it makes them better.


In [6]:
LONG_PROSE = " ".join([
    "The logistics industry has undergone considerable transformation in recent years, with",
    "many organisations investing heavily in automation and digital tracking systems in order",
    "to improve the visibility of shipments across increasingly complex supply chains. It is",
    "widely acknowledged that customers now expect a level of transparency that would have",
    "been considered unusual only a decade ago, and companies that fail to provide this",
    "transparency frequently find themselves at a competitive disadvantage relative to peers.",
] * 8)

try:
    from llmlingua import PromptCompressor
    lc = PromptCompressor(
        model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
        use_llmlingua2=True,
    )
    res = lc.compress_prompt(LONG_PROSE, rate=0.4, force_tokens=["\n", "?", ".", ","])
    print("ORIGINAL  ", ntok(LONG_PROSE), "tokens")
    print("COMPRESSED", ntok(res["compressed_prompt"]), "tokens")
    print(res["compressed_prompt"][:600], "...")
except Exception as e:
    print("LLMLingua isn't installed (that's expected unless you ran `uv sync --extra compress`).")
    print(f"  ({type(e).__name__})")
    print()
    print("Published results, for reference:")
    print("  LLMLingua      (EMNLP 2023)  up to 20x compression with little loss on its benchmarks")
    print("  LLMLingua-2    (ACL 2024)    2-5x compression, 3-6x faster than LLMLingua")
    print("  LongLLMLingua  (ACL 2024)    ~4x compression and +21.4% accuracy on long-context QA")


LLMLingua isn't installed (that's expected unless you ran `uv sync --extra compress`).
  (ModuleNotFoundError)

Published results, for reference:
  LLMLingua      (EMNLP 2023)  up to 20x compression with little loss on its benchmarks
  LLMLingua-2    (ACL 2024)    2-5x compression, 3-6x faster than LLMLingua
  LongLLMLingua  (ACL 2024)    ~4x compression and +21.4% accuracy on long-context QA


In [7]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0459


,label,model,input,output,cache_write,cache_read,usd,note
0,bloated: What is the auto-approval,claude-sonnet-5,630,164,0,0,0.002900,
1,bloated: What is the auto-approval,claude-sonnet-5,630,163,0,0,0.002890,
2,bloated: How many prior claims disq,claude-sonnet-5,632,139,0,0,0.002654,
3,bloated: How many prior claims disq,claude-sonnet-5,632,153,0,0,0.002794,
4,bloated: What format do tracking nu,claude-sonnet-5,627,94,0,0,0.002194,
5,bloated: What format do tracking nu,claude-sonnet-5,627,94,0,0,0.002194,
6,bloated: My refund is $900. What ha,claude-sonnet-5,627,159,0,0,0.002844,
7,bloated: My refund is $900. What ha,claude-sonnet-5,627,250,0,0,0.003754,
8,bloated: Can you tell me the CEO's,claude-sonnet-5,630,94,0,0,0.002200,
9,bloated: Can you tell me the CEO's,claude-sonnet-5,630,121,0,0,0.002470,


---
## What to take away

- Hand compression is free and quick, and on a prompt that has grown over time it usually gets most of the available saving.
- Always put the eval score next to the token saving. A cut you haven't tested hasn't been made safely.
- Run each question more than once. A single run can't tell a real regression from an unlucky answer.
- Compress prose. Leave code, numbers, IDs, and anything the model must reproduce exactly untouched.
- If the prompt is already cached, compression saves much less than the uncached arithmetic suggests.


### Check yourself

**1. You cut a system prompt from 1,200 to 700 tokens. It's sent on 3 million calls a month at \$2 per million input tokens, and it isn't cached. What's the monthly saving? And roughly what if it were cached at 0.1× the input price?**

<details><summary>Show answer</summary>

Uncached: 500 tokens × 3,000,000 = 1.5 billion tokens = **\$3,000 a month**. Cached, those tokens would only have cost 0.1 × \$2 per million, so the saving shrinks to about **\$300 a month**.

</details>

**2. Your compressed prompt scores 5/5 on a single run and the original scored 4/5. Can you conclude the compressed version is better?**

<details><summary>Show answer</summary>

No. With five questions and one run, a one-answer difference is well within normal run-to-run variation. Run each question several times, or use more questions, before you conclude anything beyond 'no obvious regression'.

</details>

**3. A colleague wants to run LLMLingua over the whole request, including the tool results that contain order IDs and invoice totals. What's the risk?**

<details><summary>Show answer</summary>

Token-dropping compression can remove or split characters inside IDs and numbers, so the model sees a different order ID or amount and acts on it. Compress only the prose parts and pass IDs, numbers, and code through unchanged.

</details>


### Try it on your own work

Take your longest system prompt and apply the five-step checklist. Write down 10 real questions from your logs with what a correct answer must contain, and run both versions against them two or three times each. Ship the shorter prompt only if the score holds.
